# Módulo 08 · Regresión Logística y Decisiones de Clasificación

Laboratorio reproducible: de la función sigmoide a calibración, costos, softmax y monitoreo. **Objetivo:** conectar estimación probabilística con decisiones de negocio sin confundir ranking, calibración y política.


## 1. Preparación y datos sintéticos
Generamos una muestra con señal, ruido y desbalance moderado. La semilla fija permite reproducir resultados.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (confusion_matrix, classification_report, roc_auc_score, average_precision_score,
                             log_loss, brier_score_loss, roc_curve, precision_recall_curve)
from sklearn.calibration import calibration_curve

RANDOM_STATE = 42
X, y = make_classification(n_samples=2500, n_features=8, n_informative=5, n_redundant=1,
                           weights=[0.78, 0.22], class_sep=1.25, flip_y=0.025,
                           random_state=RANDOM_STATE)
cols=[f'x{i+1}' for i in range(X.shape[1])]
df=pd.DataFrame(X,columns=cols); df['y']=y
df['y'].value_counts(normalize=True).rename('proporción')


## 2. Sigmoide, odds y logit
La regresión logística es lineal en **log-odds**, no en probabilidad.

In [ ]:
def sigmoid(z): return 1/(1+np.exp(-z))
z=np.linspace(-7,7,400); p=sigmoid(z)
fig,ax=plt.subplots(figsize=(8,4)); ax.plot(z,p); ax.axhline(.5,ls='--'); ax.axvline(0,ls='--');
ax.set(xlabel='z = Xβ',ylabel='P(Y=1|X)',title='Función sigmoide'); plt.show()

p0=.30; odds=p0/(1-p0); logit=np.log(odds)
pd.Series({'p':p0,'odds':odds,'logit':logit,'sigmoid(logit)':sigmoid(logit)})


## 3. Split honesto y pipeline
El escalado se ajusta **solo dentro del train** mediante Pipeline. El test queda sellado.

In [ ]:
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,stratify=y,random_state=RANDOM_STATE)
pipe=Pipeline([('scale',StandardScaler()),('logit',LogisticRegression(max_iter=2000))])
pipe.fit(Xtr,ytr)
prob=pipe.predict_proba(Xte)[:,1]
pd.Series({'ROC_AUC':roc_auc_score(yte,prob),'Average_Precision':average_precision_score(yte,prob),
           'LogLoss':log_loss(yte,prob),'Brier':brier_score_loss(yte,prob)})


## 4. Coeficientes y odds ratios
En variables estandarizadas, `exp(β)` es el factor multiplicativo sobre odds por un aumento de **1 desvío estándar** en la variable.

In [ ]:
coef=pipe.named_steps['logit'].coef_[0]
coef_table=pd.DataFrame({'variable':cols,'beta':coef,'odds_ratio':np.exp(coef)})
coef_table.reindex(coef_table.beta.abs().sort_values(ascending=False).index)


## 5. Efectos marginales
Para una variable continua, el efecto local sobre la probabilidad depende de `p(1-p)`. Mostramos efectos por observación y su promedio (AME).

In [ ]:
Xs=pipe.named_steps['scale'].transform(Xte)
beta=pipe.named_steps['logit'].coef_[0]
marginal=prob[:,None]*(1-prob[:,None])*beta[None,:]
ame=pd.Series(marginal.mean(axis=0),index=cols,name='AME')
ame.sort_values(key=np.abs,ascending=False)


## 6. Threshold como política
Variamos el umbral y calculamos métricas. No existe un threshold universal: depende del costo de FP/FN y de capacidad.

In [ ]:
rows=[]
for t in np.arange(.1,.91,.05):
    pred=(prob>=t).astype(int); tn,fp,fn,tp=confusion_matrix(yte,pred).ravel()
    precision=tp/(tp+fp) if tp+fp else 0; recall=tp/(tp+fn) if tp+fn else 0
    rows.append([t,tp,fp,fn,tn,precision,recall,(tp+fp)/len(pred)])
thr=pd.DataFrame(rows,columns=['threshold','TP','FP','FN','TN','precision','recall','action_rate'])
thr.head()


In [ ]:
fig,ax=plt.subplots(figsize=(8,4)); ax.plot(thr.threshold,thr.precision,label='precision'); ax.plot(thr.threshold,thr.recall,label='recall');
ax.plot(thr.threshold,thr.action_rate,label='tasa de acción'); ax.set(xlabel='threshold',ylabel='valor',ylim=(0,1)); ax.legend(); plt.show()


## 7. Decisión basada en costos
Ejemplo: FP cuesta 10 unidades y FN cuesta 80. Buscamos el threshold con menor costo sujeto a una capacidad máxima de acción del 40%.

In [ ]:
C_FP,C_FN,CAPACITY=10,80,.40
thr['cost']=thr.FP*C_FP+thr.FN*C_FN
feasible=thr[thr.action_rate<=CAPACITY]
best=feasible.loc[feasible.cost.idxmin()]
best[['threshold','precision','recall','action_rate','cost']]


## 8. ROC vs Precision–Recall
ROC resume ranking global. Precision–Recall hace visible el costo de falsos positivos cuando la clase positiva es escasa.

In [ ]:
fpr,tpr,_=roc_curve(yte,prob); prec,rec,_=precision_recall_curve(yte,prob)
fig,ax=plt.subplots(figsize=(7,5)); ax.plot(fpr,tpr,label=f'ROC AUC={roc_auc_score(yte,prob):.3f}'); ax.plot([0,1],[0,1],'--'); ax.set(xlabel='FPR',ylabel='TPR'); ax.legend(); plt.show()
fig,ax=plt.subplots(figsize=(7,5)); ax.plot(rec,prec,label=f'AP={average_precision_score(yte,prob):.3f}'); ax.axhline(yte.mean(),ls='--',label='prevalencia'); ax.set(xlabel='Recall',ylabel='Precision'); ax.legend(); plt.show()


## 9. Calibración
Comparamos probabilidad pronosticada con frecuencia observada. Un ranking fuerte puede coexistir con probabilidades mal calibradas.

In [ ]:
frac,meanp=calibration_curve(yte,prob,n_bins=10,strategy='quantile')
fig,ax=plt.subplots(figsize=(6,6)); ax.plot(meanp,frac,'o-',label='modelo'); ax.plot([0,1],[0,1],'--',label='ideal'); ax.set(xlabel='Probabilidad media predicha',ylabel='Frecuencia observada'); ax.legend(); plt.show()


## 10. Regularización y validación cruzada
Comparamos distintos valores de `C` (inverso de la fuerza de regularización) usando ROC-AUC fuera de fold.

In [ ]:
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=RANDOM_STATE)
reg=[]
for C in [.01,.03,.1,.3,1,3,10]:
    m=Pipeline([('scale',StandardScaler()),('logit',LogisticRegression(C=C,max_iter=2000))])
    sc=cross_val_score(m,Xtr,ytr,cv=cv,scoring='roc_auc')
    reg.append([C,sc.mean(),sc.std()])
pd.DataFrame(reg,columns=['C','cv_auc_mean','cv_auc_sd'])


## 11. Softmax: extensión multiclase
Softmax convierte K logits en probabilidades que suman 1.

In [ ]:
def softmax(z,temperature=1.0):
    z=np.asarray(z)/temperature; e=np.exp(z-z.max()); return e/e.sum()
logits=np.array([2.0,1.0,0.0])
pd.DataFrame({'clase':['A','B','C'],'T=0.5':softmax(logits,.5),'T=1':softmax(logits,1),'T=2':softmax(logits,2)})


## 12. Monitoring conceptual
Simulamos una población futura con menor separación entre clases. El objetivo es mostrar que el desempeño debe medirse nuevamente en producción.

In [ ]:
X_new,y_new=make_classification(n_samples=1500,n_features=8,n_informative=5,n_redundant=1,weights=[.72,.28],class_sep=.75,flip_y=.04,random_state=99)
p_new=pipe.predict_proba(X_new)[:,1]
pd.DataFrame({'métrica':['prevalencia','ROC-AUC','AP','Brier'],
              'test_original':[yte.mean(),roc_auc_score(yte,prob),average_precision_score(yte,prob),brier_score_loss(yte,prob)],
              'producción_simulada':[y_new.mean(),roc_auc_score(y_new,p_new),average_precision_score(y_new,p_new),brier_score_loss(y_new,p_new)]})


## Cierre
Una evaluación defendible separa cinco preguntas: **(1)** ¿el modelo generaliza?, **(2)** ¿ordena bien?, **(3)** ¿sus probabilidades son confiables?, **(4)** ¿qué threshold/política maximiza valor bajo restricciones?, **(5)** ¿esas propiedades se mantienen en producción?
